In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run /Workspace/fmcg_domain/databricks_fmcg_data_engineering/Setup/utilities

In [0]:
bronze_schema,silver_schema,gold_schema
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","customers","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

In [0]:
import os 
s3_path = f"s3://databrickssaugat/{data_source}"

landing_path = f"{s3_path}/landing"
processed_path = f"{s3_path}/processed"

print("base path: ",s3_path)
print("landing path: ",landing_path)
print("processed path: ",processed_path)

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.staging_{data_source}"

In [0]:
df = spark.read.options(
    header = True,
    inferSchema=True
).csv(f"{landing_path}/*.csv").withColumn("ingest_timestamp", current_timestamp()).select(
    "*","_metadata.file_name","_metadata.file_size"
)

display(df)


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .option("delta.enableChangeDataFeed","true")\
            .saveAsTable(bronze_table)


In [0]:
files = [x.file_name for x in df.select("file_name").distinct().collect()]

for file in files:
    
    dbutils.fs.mv(f"{landing_path}/{file}", f"{processed_path}/{file}")